# Batch BH fitting workflow

Run BH fits for an explicit list of `SHOTS` from a data folder.

Preprocessing for fitting is the shared `prepare_bh_fit_arrays` pipeline:

- subtract the mean of `BACKGROUND_FRAMES` for each (frame, channel),
- crop to `BH_FIT_WAVELENGTH_RANGE_NM`,
- compute the normalization scale strictly inside `BH_SCALE_WAVELENGTH_RANGE_NM`,
- divide by that scale (negatives preserved).

The fitter's constant baseline `base` is tightly bounded near zero by default.

`run_folder_batch` resumes by skipping shots whose `summary.csv` already exists in `OUT_DIR/<shot>/`.

In [ ]:
from pathlib import Path

from bh_molecule import run_folder_batch

DATA_DIR = Path("~/Dropbox/Experiments/2025-LHD-BH/133mORCA").expanduser()
OUT_DIR = Path("bh_batch_results")

SHOTS = [
    193788,
    193789,
    193790,
]

BACKGROUND_FRAMES = (0, 1, 2, 3)
BH_FIT_WAVELENGTH_RANGE_NM = (433.05, 433.90)
BH_SCALE_WAVELENGTH_RANGE_NM = (433.08, 433.30)

SAVE_FRAMES = False
RUN_FIT_LIMIT = None

CW_NM = 431.91
SCALE = 1.0
TIME_RANGE = (0.0, 10.0)

FRAMES = None
CHANNELS = None

# --- Explicit, reproducible fitter constraints -------------------------------
# Use the calibration notebook (`examples/14_w_inst_calibration.ipynb`) on
# representative spectra to choose `W_INST_DEFAULT` and `W_INST_BOUNDS`.
# Set any of these to `None` to keep the package defaults.
#
# W_INST_DEFAULT  : initial guess (and the fixed value when FIX_W_INST=True)
#                   for the instrumental Gaussian FWHM [nm].
# W_INST_BOUNDS   : tight production bounds [nm]; must satisfy 0 <= lo < hi.
# FIX_W_INST      : if True, w_inst is fixed at W_INST_DEFAULT via parameter
#                   elimination (curve_fit sees only 6 free parameters).
# DX_TOL_NM       : half-width of the allowed wavelength shift dx [nm].
# BASE_BOUND      : half-width of the tight `base` bound applied after
#                   preprocessing.
W_INST_DEFAULT = None
W_INST_BOUNDS = None
FIX_W_INST = False
DX_TOL_NM = None
BASE_BOUND = None

In [ ]:
out_dir = OUT_DIR if OUT_DIR.is_absolute() else (Path.cwd() / OUT_DIR).resolve()

all_fits = sorted(DATA_DIR.glob("*.fits"))
if SHOTS is None:
    selected = all_fits
else:
    wanted = {str(s) for s in SHOTS}
    selected = [p for p in all_fits if p.stem in wanted]
    missing = wanted - {p.stem for p in selected}
    if missing:
        print(f"WARNING: shots {sorted(missing)} not found in {DATA_DIR}")

print(f"Selected data folder:  {DATA_DIR}")
print(f"Output folder:         {out_dir}")
print(f"Selected shots:        {SHOTS}")
print(f"Matched FITS files     ({len(selected)} of {len(all_fits)} total in folder):")
for p in selected:
    print(f"  - {p}")
print(f"Background frames:     {BACKGROUND_FRAMES}")
print(f"BH fit window:         {BH_FIT_WAVELENGTH_RANGE_NM} nm")
print(f"BH scale window:       {BH_SCALE_WAVELENGTH_RANGE_NM} nm")
print(f"SAVE_FRAMES:           {SAVE_FRAMES}")
if SAVE_FRAMES:
    print(
        "  WARNING: SAVE_FRAMES=True is slower and writes per-fit PNGs to "
        f"<out_dir>/<shot>/frames/"
    )
print(f"RUN_FIT_LIMIT:         {RUN_FIT_LIMIT}")
print("--- fitter constraints (None = package default) ---")
print(f"W_INST_DEFAULT:        {W_INST_DEFAULT}")
print(f"W_INST_BOUNDS:         {W_INST_BOUNDS}")
print(f"FIX_W_INST:            {FIX_W_INST}")
print(f"DX_TOL_NM:             {DX_TOL_NM}")
print(f"BASE_BOUND:            {BASE_BOUND}")

In [ ]:
results = run_folder_batch(
    DATA_DIR,
    frames=FRAMES,
    channels=CHANNELS,
    shots=SHOTS,
    cw=CW_NM,
    scale=SCALE,
    time_range=TIME_RANGE,
    background_frames=BACKGROUND_FRAMES,
    bh_fit_range=BH_FIT_WAVELENGTH_RANGE_NM,
    bh_scale_range=BH_SCALE_WAVELENGTH_RANGE_NM,
    w_inst_default=W_INST_DEFAULT,
    w_inst_bounds=W_INST_BOUNDS,
    fix_w_inst=FIX_W_INST,
    dx_tol_nm=DX_TOL_NM,
    base_bound=BASE_BOUND,
    out_dir=out_dir,
    save_frames=SAVE_FRAMES,
    run_fit_limit=RUN_FIT_LIMIT,
)
print(f"\nFinished {len(results)} shot(s).")
list(results.keys())

In [ ]:
if results:
    first_shot_id = sorted(results.keys())[0]
    print(f"summary.csv for {first_shot_id} -> {out_dir / first_shot_id / 'summary.csv'}")
    display(results[first_shot_id].head())
else:
    print("No new shots were fit (all selected shots already have summary.csv).")